# Blinkit Delhi-NCR Replenishment: Heterogeneous Fleet Capacitated VRP

**Problem.** Every night a single mother warehouse (Okhla, New Delhi) replenishes **361 Blinkit dark stores** across Delhi and core NCR (Gurugram, Noida, Greater Noida, Ghaziabad, Faridabad). Each store has a replenishment requirement from the forecasting team and a limited free storage space. A mixed fleet of **3 vehicle types** (Tata Ace, Tata 407, Eicher 14 ft) is available. Each type has a different capacity, fixed cost and per-km cost, and some stores in dense markets cannot be reached by larger trucks.

**Decision.** Which vehicles to dispatch, which stores each vehicle serves, and in which order.

**Goal.** Minimise total cost = fixed vehicle cost + distance-based operating cost, so that dispatched vehicles are as full as possible.

**Problem class.** Heterogeneous Fixed Fleet Capacitated Vehicle Routing Problem with site-dependent (vehicle–store compatibility) constraints, on an **asymmetric real road network** (OSRM).

Data sources and generation are documented in [DATA.md](DATA.md).

## 1. Assumptions

1. **Single depot** with unlimited stock. Every route starts and ends at the warehouse.
2. **Deterministic demand:** the forecast is known before planning and treated as exact. Forecasting itself is out of scope; the forecast file is an input.
3. **Single visit:** each store with positive demand is visited **exactly once** by **one** vehicle (no split deliveries).
4. **One trip per vehicle** per planning night.
5. **No time dimension:** no time windows, no loading/unloading times, no route duration limits. OSRM drive times are reported only as a KPI.
6. **Capacity in crates:** all goods move in standard crates. Volume, not weight, is the binding capacity.
7. **Linear cost:** operating cost = cost per km × OSRM road distance. Fixed cost is charged once per dispatched vehicle.
8. **Asymmetric distances:** $d_{ij} \neq d_{ji}$ is allowed (one-ways, flyovers, divided roads).
9. **Homogeneous vehicles within a type:** all vehicles of a type have identical capacity and cost.
10. **Store access:** a store can only be served by vehicle types no larger than its `max_vehicle_type`.
11. **Effective demand** is capped by free storage space and by the largest allowed vehicle (see Section 2). Any excess is reported as *unserved*, not forced.
12. Stores with $q_i = 0$ (no space or no requirement) are dropped from the instance.

## 2. Notation

### Sets

| Symbol | Meaning |
|---|---|
| $0$ | Depot (mother warehouse) |
| $S = \{1, \dots, n\}$ | Dark stores with $q_i > 0$ |
| $N = \{0\} \cup S$ | All nodes |
| $K = \{S, M, L\}$ | Vehicle types (small, medium, large) |
| $A = \{(i,j) : i, j \in N,\ i \neq j\}$ | Arcs of the complete directed road graph |

### Parameters

| Symbol | Meaning | Source |
|---|---|---|
| $d_{ij}$ | Road distance from $i$ to $j$ (km) | OSRM `distance_km.csv` |
| $Q_k$ | Capacity of a type-$k$ vehicle (crates) | `vehicles.csv` |
| $F_k$ | Fixed cost of dispatching one type-$k$ vehicle (₹) | `vehicles.csv` |
| $c_k$ | Operating cost of type $k$ (₹/km) | `vehicles.csv` |
| $m_k$ | Number of type-$k$ vehicles available | `vehicles.csv` |
| $a_{jk}$ | $1$ if store $j$ can be served by type $k$, else $0$ | `store_params.csv` |
| $R_j$ | Forecast replenishment requirement (crates) | `forecast_<scenario>.csv` |
| $C_j,\ H_j$ | Storage capacity and on-hand stock (crates) | `store_params.csv` |
| $q_j$ | Effective demand (crates), $q_0 = 0$ | Derived |

Effective demand, so the model adapts automatically to any forecast:

$$
q_j \;=\; \min\Big(R_j,\;\; C_j - H_j,\;\; \max_{k:\,a_{jk}=1} Q_k\Big) \qquad \forall j \in S
$$

### Decision Variables

| Symbol | Domain | Meaning |
|---|---|---|
| $x_{ijk}$ | $\{0,1\}$ | $1$ if a type-$k$ vehicle travels directly from $i$ to $j$ |
| $f_{ijk}$ | $\mathbb{R}_{\ge 0}$ | Load (crates) on board a type-$k$ vehicle while travelling arc $(i,j)$ |

The number of type-$k$ vehicles dispatched is $\sum_{j \in S} x_{0jk}$.

## 3. Mathematical Model

### Objective: minimise total replenishment cost

$$
\min \; Z \;=\; \underbrace{\sum_{k \in K} F_k \sum_{j \in S} x_{0jk}}_{\text{fixed cost of vehicles used}} \;+\; \underbrace{\sum_{k \in K} \sum_{(i,j) \in A} c_k\, d_{ij}\, x_{ijk}}_{\text{distance-based operating cost}}
$$

### Constraints

$$
\begin{aligned}
&(1)\quad \sum_{k \in K} \sum_{i \in N,\, i \neq j} x_{ijk} = 1 && \forall\, j \in S \\[4pt]
&(2)\quad \sum_{i \in N,\, i \neq j} x_{ijk} = \sum_{l \in N,\, l \neq j} x_{jlk} && \forall\, j \in N,\ k \in K \\[4pt]
&(3)\quad \sum_{j \in S} x_{0jk} \le m_k && \forall\, k \in K \\[4pt]
&(4)\quad \sum_{i \in N,\, i \neq j} x_{ijk} \le a_{jk} && \forall\, j \in S,\ k \in K \\[4pt]
&(5)\quad \sum_{k \in K} \sum_{i \in N,\, i \neq j} f_{ijk} \;-\; \sum_{k \in K} \sum_{l \in N,\, l \neq j} f_{jlk} = q_j && \forall\, j \in S \\[4pt]
&(6)\quad q_j\, x_{ijk} \;\le\; f_{ijk} \;\le\; (Q_k - q_i)\, x_{ijk} && \forall\, (i,j) \in A,\ k \in K \\[4pt]
&(7)\quad f_{i0k} = 0 && \forall\, i \in S,\ k \in K \\[4pt]
&(8)\quad x_{ijk} \in \{0,1\},\quad f_{ijk} \ge 0 && \forall\, (i,j) \in A,\ k \in K
\end{aligned}
$$

### Meaning of each constraint

| # | Name | Interpretation |
|---|---|---|
| (1) | **Single visit** | Every dark store is entered exactly once, by exactly one vehicle of one type. |
| (2) | **Route continuity (per type)** | A vehicle of type $k$ that enters a node also leaves it as type $k$. This keeps a whole route on one vehicle type. At the depot it means every dispatched vehicle returns. |
| (3) | **Fleet availability** | No more type-$k$ vehicles dispatched than are available. |
| (4) | **Vehicle–store compatibility** | A type-$k$ vehicle may enter store $j$ only if it can physically reach and unload there (narrow-lane markets). By (2), it then also cannot leave via $j$. |
| (5) | **Load flow balance** | The load on board drops by exactly $q_j$ at store $j$, so each store receives its full effective demand. |
| (6) | **Capacity and linking** | Load can travel on arc $(i,j)$ only if that arc is used. The load leaving the depot never exceeds $Q_k$. On arrival at $j$ the truck still carries at least $q_j$. |
| (7) | **Return empty** | Vehicles return to the warehouse with no load, so the load leaving the depot equals exactly the route's total demand. |
| (8) | **Domains** | Binary routing, non-negative load. |

### Why there are no subtours
Constraints (5)–(6) are a **single-commodity flow** formulation. Along any route the on-board load strictly decreases ($q_j \ge 1$ at every store). A cycle that does not pass through the depot would need the load to decrease all the way around and return to its starting value, which is impossible. Every route is therefore forced to start from the depot. No exponential family of subtour-elimination cuts is needed, so the model stays compact.

### Why minimising cost maximises utilisation
The fixed cost $F_k$ is paid per vehicle regardless of load. Sending a half-empty truck wastes part of $F_k$, so the cost-optimal solution **consolidates loads onto as few, as full, vehicles as possible**. The model uses smaller or extra vehicles only when the extra distance cost or the access restrictions justify it. Utilisation is therefore an *outcome* of the objective, not a competing goal.

## 4. Performance Indicators (reported after solving)

For a route $r$ operated by a vehicle of type $k(r)$ visiting store set $S_r$:

$$
\text{Load}_r = \sum_{j \in S_r} q_j, \qquad
U_r = \frac{\text{Load}_r}{Q_{k(r)}} \times 100\%, \qquad
\text{Distance}_r = \sum_{(i,j) \in r} d_{ij}
$$

$$
\text{Fleet utilisation} = \frac{\sum_{j \in S} q_j}{\sum_{r} Q_{k(r)}} \times 100\%, \qquad
\text{Cost per crate} = \frac{Z}{\sum_{j \in S} q_j}
$$

Additional KPIs: vehicles used by type, fixed vs. variable cost split, total km, total drive time (OSRM), unserved crates $\sum_j (R_j - q_j)$.

## 5. Model Size

With $n$ stores and $|K| = 3$ types, the full model has $|K| \cdot (n+1) \cdot n$ binary arc variables. Two pre-processing reductions are applied before solving, without changing the optimal solution:

- **Compatibility (4):** a type-$k$ vehicle gets no arcs into or out of a store with $a_{jk} = 0$.
- **Capacity:** a type-$k$ vehicle gets no arcs touching a store with $q_j > Q_k$. Constraint (6) already forces those arcs to 0.

Flow variables exist only for arcs that do not end at the depot, by constraint (7). The cells below load the data and compute the exact size of the instance.

In [ ]:
# reload edited src/ modules automatically, no kernel restart needed
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from common import effective_demand, load_config, vehicles_table

cfg = load_config()
# change this to re-plan with another forecast, e.g. "data/synthetic/forecasts/forecast_festive.csv"

#FORECAST_FILE = "data/synthetic/forecasts/forecast_festive.csv"   # or forecast_low.csv
FORECAST_FILE = cfg["model"]["forecast_file"]

vehicles = vehicles_table(cfg)
store_params = pd.read_csv(ROOT / "data/synthetic/store_params.csv")
forecast = pd.read_csv(ROOT / FORECAST_FILE)

demand = effective_demand(store_params, forecast, vehicles)
demand = demand[demand["demand_crates"] > 0]
print(f"Forecast file: {FORECAST_FILE}")
vehicles

In [ ]:
cap = vehicles.set_index("type")["capacity_crates"]
allowed = pd.DataFrame(
    {
        k: (cap[k] <= demand["max_vehicle_type"].map(cap)) & (demand["demand_crates"] <= cap[k])
        for k in vehicles["type"]
    }
)
n = len(demand)
# per type: depot + allowed stores form a complete directed graph
x_vars = int(sum((c + 1) * c for c in allowed.sum()))
f_vars = int(sum(c * c for c in allowed.sum()))

summary = pd.Series(
    {
        "Stores in model (n)": n,
        "Total forecast R (crates)": int(demand["forecast_crates"].sum()),
        "Total effective demand q (crates)": int(demand["demand_crates"].sum()),
        "Unserved (crates)": int(demand["unserved_crates"].sum()),
        "Fleet capacity (crates)": int((vehicles["capacity_crates"] * vehicles["available"]).sum()),
        "Lower bound on vehicles (all L)": int(np.ceil(demand["demand_crates"].sum() / cap.max())),
        "Binary arc variables x_ijk": x_vars,
        "Continuous flow variables f_ijk": f_vars,
    }
)
summary.to_frame("value")

## 6. Solution Approach

The HFVRP is NP-hard. At 361 stores the MILP has about 370,000 binary variables, so no solver can prove optimality from scratch in reasonable time. A single **two-stage pipeline** is used to get the best solution together with a proven quality guarantee:

| Stage | Tool | Role |
|---|---|---|
| 1. Warm start | **Google OR-Tools** routing solver: Christofides construction + Guided Local Search | Quickly finds a high-quality feasible plan for the exact same problem (heterogeneous fleet, capacity, compatibility, fixed + per-km cost) |
| 2. Exact optimisation | **Gurobi 13** on the MILP of Section 3, MIP start = Stage 1 plan | Searches for better plans (branch-and-bound + MIP heuristics) and computes a **lower bound** on the true optimum |

The output is the best plan found by Gurobi, which is never worse than the warm start, together with its **optimality gap**:

$$
\text{Gap} = \frac{Z_{\text{best}} - Z_{\text{bound}}}{Z_{\text{best}}} \times 100\%
$$

The true optimal cost is guaranteed to lie between $Z_{\text{bound}}$ and $Z_{\text{best}}$.

### Project code layout

| Folder | Contents |
|---|---|
| [src/data_pipeline/](src/data_pipeline/) | Store filtering, synthetic data, OSRM matrices. Run everything with `.venv\Scripts\python.exe src\data_pipeline\run_all.py` |
| [src/solver/](src/solver/) | `instance.py` (model input), `solve_ortools.py` (Stage 1), `solve_gurobi.py` (Stage 2), `results.py` (save outputs) |
| [src/analysis/](src/analysis/) | `report.py` (KPIs, charts), `route_map.py` (folium map on OSRM roads) |

Time limits and the Gurobi licence path are set in [config.yaml](config.yaml) under `solver`. With the default limits (300 s + 1,200 s) the two stages below take about 25 minutes. Results are written to `outputs/<scenario>/`.

In [ ]:
import time

from solver.instance import load_instance, route_table
from solver.results import save_results, scenario_dir
from solver.solve_gurobi import solve_gurobi
from solver.solve_ortools import solve_ortools

scfg = cfg["solver"]
inst = load_instance(cfg, FORECAST_FILE)
OUT = scenario_dir(cfg, inst)
print(f"Scenario: {OUT.name} | stores: {len(inst.q) - 1} | demand: {inst.q.sum()} crates | output: {OUT.relative_to(ROOT)}")

### Stage 1: OR-Tools warm start

In [ ]:
t0 = time.perf_counter()
start_routes = solve_ortools(inst, scfg["ortools_time_limit_sec"], first_solution=scfg["ortools_first_solution"])
ortools_runtime = time.perf_counter() - t0

warm = route_table(inst, start_routes)
print(f"Warm start: {len(warm)} vehicles {warm['vehicle_type'].value_counts().to_dict()} | "
      f"cost Rs {warm['total_cost_inr'].sum():,.0f} | avg utilisation {warm['utilization_pct'].mean():.1f}% | "
      f"{ortools_runtime:.0f}s")

### Stage 2: Gurobi MILP (warm-started) and saving results

The Gurobi log below shows the MIP start being loaded, the root LP bound, cutting planes and the final gap. The full log is also saved to `outputs/<scenario>/gurobi.log`.

In [ ]:
final_routes, gurobi_info = solve_gurobi(inst, start_routes, scfg["gurobi"], OUT / "gurobi.log")
result = save_results(inst, OUT, start_routes, final_routes, gurobi_info, ortools_runtime)

print(f"\nFinal plan: {result['routes']} vehicles | cost Rs {result['total_cost_inr']:,.0f} | "
      f"gap {gurobi_info['mip_gap_pct']:.2f}% ({gurobi_info['status']}) | saved to {OUT.relative_to(ROOT)}")

## 7. Results

### 7.1 Solution quality and headline KPIs

In [ ]:
from analysis.report import dispatch_sheet, fleet_mix, kpi_table, load_results, plot_utilization_and_cost

# reads the saved files, so the analysis also works later without re-solving
result, routes, stops = load_results(OUT)
kpi_table(result)

### 7.2 Fleet mix: which vehicle type does what

In [ ]:
fleet_mix(routes, vehicles)

### 7.3 Capacity utilisation and cost structure

In [ ]:
fig = plot_utilization_and_cost(routes, vehicles)

### 7.4 Route plan (dispatch sheet)

In [ ]:
dispatch_sheet(routes)

## 8. Route Map

Each route is drawn on the **actual road path** returned by the OSRM Route service.

- **Map style:** use the layer control (top-right) to switch the base map between **Carto Positron (light)**, **Carto Voyager**, **Carto Dark Matter**, **Esri Street Map** and **Esri Satellite**.
  - Carto styles use `CARTO_API_KEY` from [config.yaml](config.yaml).
  - Esri styles need no key.
  - OpenStreetMap's own tiles are not used: they show "Access blocked" for maps opened from a local file or a notebook.
- **Layers:** routes can be toggled per vehicle type, and the red dashed circles (off by default) are the restricted-access zones.
- **Details:** click a store to see its route, stop number and delivery. Hover over a route for its load, distance and cost.

Road paths are cached per route in `outputs/<scenario>/route_geometries.json`, so only changed routes are re-fetched after a re-solve. The map is also saved as `outputs/<scenario>/route_map.html`.

In [ ]:
from analysis.route_map import build_map

build_map(cfg, OUT)

## 9. Key Findings

The findings below are **generated from the saved results** of the current scenario (`outputs/<scenario>/`). They stay correct when the data, configuration or forecast changes.

In [ ]:
from IPython.display import Markdown

from analysis.report import key_findings

Markdown(key_findings(OUT, vehicles))